In [1]:
import os
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["XLA_PYTHON_CLIENT_ALLOCATOR"] = "platform"

import superstats as sup
import bayesflow as bf
import numpy as np
import jax

print(jax.devices())

INFO:jax._src.xla_bridge:Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: dlopen(libtpu.so, 0x0001): tried: 'libtpu.so' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibtpu.so' (no such file), '/Users/lschumacher/.local/share/uv/python/cpython-3.13.14-macos-aarch64-none/lib/libtpu.so' (no such file), '/usr/lib/libtpu.so' (no such file, not in dyld cache), 'libtpu.so' (no such file)
INFO:bayesflow:Using backend 'jax'


[CpuDevice(id=0)]


/Users/lschumacher/Documents/GitHub/superstats/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Specify Prior

In [ ]:
joint_prior = sup.prior.JointPrior(
    v = sup.transition.RandomWalk(
            bounds=(-1.0, 1.0),
            sigma=sup.prior.Prior(dist="halfnormal", scale=0.1),
            delta=sup.prior.Prior(dist="halfnormal", scale=0.02),
            initial_prior=sup.prior.Prior(dist="normal", low=0.0, high=1.0)
        ),
    a = sup.transition.Mixture(
        transitions=[
            sup.transition.AutoRegression(),
            sup.transition.Jump()
        ],
        mixture_weights=sup.prior.Prior(dist="dirichlet", alpha=[25.0, 1.5]),
        bounds=(0.0, 4.0)
    ),
    bias = 0.5,
    tau = sup.prior.Prior(dist="halfnormal", scale=0.15)
)

In [ ]:
fig = joint_prior.plot_time_varying_prior(
    num_steps=300,
    num_trajectories=10
)

In [ ]:
fig = joint_prior.plot_time_invariant_prior(num_cols=2)

In [ ]:
fig = joint_prior.plot_joint_prior()

## Specify Generative Model

In [ ]:
ddm = sup.simulation.sample_ddm

In [ ]:
generative_model = sup.simulation.GenerativeModel(
    prior=joint_prior,
    model=ddm,
)

In [ ]:
plot = generative_model.plot_push_forward(
    num_sim=10,
    num_steps=200,
    data_dim=0,
    kind="dist",
    # aggregation=np.median,
    uncertainty_fun="95ci",
    marginal=True,
    # spaghetti=False,
    # num_cols=5,
    # figsize=(15, 6)
)


## Specify Workflow

In [ ]:
workflow = sup.workflow.Workflow(
    simulator=generative_model,
    checkpoint_filepath="basic_ddm"
)

### Offline Training

In [ ]:
train_data = workflow.simulator.sample(
    batch_size=10,
    num_steps=100,
    tile_to_steps=True
)
test_data = workflow.simulator.sample(
    batch_size=2,
    num_steps=100,
    tile_to_steps=True
)

In [ ]:
history = workflow.fit_offline(
    data=train_data,
    validation_data=test_data,
    epochs=2,
    batch_size=8
)

In [ ]:
plot = workflow.plot_history(history)

## Online Training

In [ ]:
history = workflow.fit_online(
    num_steps=300,
    epochs=75,
    num_batches_per_epoch=500,
    batch_size=32
)

In [ ]:
plot = workflow.plot_history(history)

## Validation

In [ ]:
val_data = workflow.simulator.sample(
    batch_size=10,
    num_steps=100
)

In [ ]:
samples = workflow.sample(
    data=val_data['data'],
    num_samples=50,
)

### Time-varying Parameters

In [ ]:
fig = workflow.verify_time_varying(val_data, samples)

### Time-invariant Parameters

In [ ]:
fig_recovery, fig_calibration = workflow.verify_time_invariant(
    val_data,
    samples
)

## Data Fitting

In [ ]:
data = workflow.simulator.sample(5, 100)

In [ ]:
samples = workflow.sample(data['data'])

### Posterior Estimates

In [ ]:
local_keys = workflow.simulator.local_keys

fig = workflow.plot_time_varying_posterior(
    estimates=samples,
    targets=data,
    smoothing="ema",
    aggregation=np.median,
)

In [ ]:
fig = workflow.plot_time_varying_posterior(
    estimates=samples,
    targets=data,
    aggregation=np.median,
    aggregate_strategy="full_uncertainty",
    uncertainty_fun="95ci",
    smoothing="sma",
    smoothing_window=5,
    marginal=True,
    # num_cols=2
)

In [ ]:
fig = workflow.plot_time_invariant_posterior(
    estimates=samples,
    targets=data,
    # aggregation=np.median
    # figsize=(10, 8)
)

### Posterior Re-simulation

In [ ]:
## do two different bands with different alphas
## dashed line for real data

In [ ]:
pred_data = workflow.resimulate_posterior(samples, num_sims=10)


In [2]:
fig = sup.diagnostics.plots.plot_posterior_resimulation(
    pred_data,
    data['data'],
    data_dim=0,
    kind="trajectory",
    aggregation=np.mean,
    aggregate_strategy="full_uncertainty",
    uncertainty_fun="95ci",
    spaghetti=True,
    # marginal=True,
    smoothing="sma",
    smoothing_window=2
)

NameError: name 'pred_data' is not defined